In [38]:
import os
import json
import shutil
from sklearn.model_selection import train_test_split

In [39]:
JSON_FILE_PATH = "C:\\Dokumentumok\\hello\\pneumonia_detector_ML_project\\pneumonia_detection_CNN\\pneumonia-challenge-annotations-adjudicated-kaggle_2018.json"
SOURCE_IMAGES_DIR = "C:\\rx\\pneumonia-challenge-dataset-adjudicated-kaggle_2018\\mdai_rsna_project_x9N20BZa_images_2018-07-20-153330"
OUTPUT_BASE_DIR = "data"

LABEL_PNEUMONIA   = "L_v8n"           # Lung opacity
LABEL_NORMAL      = "L_o8w"           # Normal
LABEL_NO_OPACITY  = "L_yd0"           # No lung opacity / Not normal
NOT_PNEUMONIA_IDS = {LABEL_NORMAL, LABEL_NO_OPACITY}
 
TRAIN_RATIO = 0.70
VAL_RATIO   = 0.15
TEST_RATIO  = 0.15
RANDOM_SEED = 42
 
COPY_FILES = True

In [40]:
with open(JSON_FILE_PATH, 'r', encoding='utf-8') as f:
    data = json.load(f)

In [41]:
annotations = data["datasets"][0]["annotations"]
print(f"  Total annotations in file : {len(annotations):,}")

  Total annotations in file : 83,809


In [ ]:
# study_records[StudyInstanceUID] = {
#     "sop"    : SOPInstanceUID    (used to locate the .dcm file name)
#     "series" : SeriesInstanceUID (used to navigate the nested DICOM folder layer)
#     "binary" : 1 (Pneumonia) | 0 (Not pneumonia) | None (not yet set)
# }

# }
 
study_records = {}
 
for ann in annotations:
    uid = ann.get("StudyInstanceUID")
    lid = ann.get("labelId")
    if not uid or not lid:
        continue
 
    if uid not in study_records:
        study_records[uid] = {"sop": None, "series": None, "binary": None}
 
    rec = study_records[uid]
 
    # Always update sop/series from the calculated label annotation itself
    if lid == LABEL_PNEUMONIA:
        rec["binary"] = 1
        rec["sop"]    = ann.get("SOPInstanceUID")
        rec["series"] = ann.get("SeriesInstanceUID")
    elif lid in NOT_PNEUMONIA_IDS and rec["binary"] != 1:
        rec["binary"] = 0
        rec["sop"]    = ann.get("SOPInstanceUID")
        rec["series"] = ann.get("SeriesInstanceUID")

 
# Drop the studies that only have non-calculated annotations
classified = {uid: rec for uid, rec in study_records.items()
              if rec["binary"] is not None}

all_ids    = list(classified.keys())
all_labels = [classified[uid]["binary"] for uid in all_ids]
 
print("\n-----------------------------------------------------------")
print("Classification (calculated labels only):")
print("-----------------------------------------------------------")
print(f"  Pneumonia     (1) : {all_labels.count(1):>6,}")
print(f"  Not pneumonia (0) : {all_labels.count(0):>6,}")
print(f"  -----------------------------------")
print(f"  Total classified  : {len(all_ids):>6,}")
print(f"  Skipped           : {30000 - len(classified):>6,}  (no calculated label)")
print("-----------------------------------------------------------\n")



-----------------------------------------------------------
Classification (calculated labels only):
-----------------------------------------------------------
  Pneumonia     (1) :  7,106
  Not pneumonia (0) : 22,578
  -----------------------------------
  Total classified  : 29,684
  Skipped           :    316  (no calculated label)
-----------------------------------------------------------



In [43]:
train_ids, temp_ids, train_labels, temp_labels = train_test_split(
    all_ids, all_labels, 
    test_size=(VAL_RATIO + TEST_RATIO), 
    stratify=all_labels, 
    random_state=RANDOM_SEED
)

val_test_ratio = TEST_RATIO / (VAL_RATIO + TEST_RATIO) 
val_ids, test_ids, val_labels, test_labels = train_test_split(
    temp_ids, temp_labels, 
    test_size=val_test_ratio, 
    stratify=temp_labels, 
    random_state=RANDOM_SEED
)

print(f"  Train set size: {len(train_ids):>6,} images")
print(f"  Val set size:   {len(val_ids):>6,} images")
print(f"  Test set size:  {len(test_ids):>6,} images")

  Train set size: 20,778 images
  Val set size:    4,453 images
  Test set size:   4,453 images


In [44]:
# Indexing files on disk using extended paths
disk_file_map = {}

for root, dirs, filenames in os.walk(SOURCE_IMAGES_DIR):
    for f in filenames:
        if f.endswith(".dcm"):
            sop_id = f[:-4]  # Extract SOPInstanceUID from filename by dropping '.dcm'
            
            # Convert to absolute path and prepend the extended path prefix '\\?\'
            full_path = os.path.abspath(os.path.join(root, f))
            if not full_path.startswith("\\\\?\\"):
                full_path = "\\\\?\\" + full_path
                
            disk_file_map[sop_id] = full_path

print(f"  Successfully indexed {len(disk_file_map):,} files from storage.")

  Successfully indexed 30,000 files from storage.


In [45]:
# Create directory structure and copy files using the nested path
label_to_folder = {0: "not_pneumonia", 1: "pneumonia"}
splits_dict = {
    "train": (train_ids, train_labels),
    "val":   (val_ids, val_labels),
    "test":  (test_ids, test_labels)
}

In [46]:

if COPY_FILES:
    copied_count = 0
    missing_files = []

    for split_name, (split_ids, split_labels) in splits_dict.items():
        for uid, label in zip(split_ids, split_labels):
            
            # Setup output folder paths
            folder_name = label_to_folder[label]
            target_dir = os.path.abspath(os.path.join(OUTPUT_BASE_DIR, split_name, folder_name))
            
            # Apply extended path prefix to destination directory to avoid safe execution limits
            if not target_dir.startswith("\\\\?\\"):
                target_dir = "\\\\?\\" + target_dir
            os.makedirs(target_dir, exist_ok=True)

            sop = classified[uid]["sop"]
            
            # Retrieve the pre-indexed long path from our map
            source_file = disk_file_map.get(sop)

            if source_file and os.path.exists(source_file):
                target_file = os.path.join(target_dir, f"{sop}.dcm")
                shutil.copy2(source_file, target_file)
                copied_count += 1
            else:
                missing_files.append(uid)

            if copied_count % 2000 == 0 and copied_count > 0:
                print(f"  - Copied {copied_count:,} files")

    print(f"\nSuccessfully copied {copied_count:,} files.")
    if missing_files:
        print(f"Warning: {len(missing_files)} files were missing or unreadable due to system permissions.")

  - Copied 2,000 files
  - Copied 4,000 files
  - Copied 6,000 files
  - Copied 8,000 files
  - Copied 10,000 files
  - Copied 12,000 files
  - Copied 14,000 files
  - Copied 16,000 files
  - Copied 18,000 files
  - Copied 20,000 files
  - Copied 22,000 files
  - Copied 24,000 files
  - Copied 26,000 files
  - Copied 28,000 files

Successfully copied 29,684 files.
